### Chapter 3.5 - Concise Implementation of Linear Regression

The concise implementation uses PyTorch objects for the model, loss, optimizer, and data loader. The goal is to map each shortcut back to the manual training loop.


In [ ]:
import math
import random
import numpy as np
import torch


### 3.5.1 Defining the Model

#### 1. Intuition

A concise model uses a framework layer instead of manually storing `w` and `b`.

`torch.nn.Linear` is a PyTorch layer that computes a linear transformation: input times weights plus bias.

#### 2. Why this exists

Framework layers reduce boilerplate and automatically register parameters so optimizers can find them.


#### 3. Examples

Create a one-output linear layer for two input features.


In [ ]:
net = torch.nn.Linear(2, 1)
X = torch.tensor([[1.0, 2.0], [3.0, 4.0]])
y_hat = net(X)
y_hat.shape


Inspect the parameter shapes.


In [ ]:
weight_shape = net.weight.shape
bias_shape = net.bias.shape
weight_shape, bias_shape


#### 4. Step-by-step breakdown

`torch.nn.Linear(2, 1)` expects 2 input features and produces 1 output per example.

`net(X)` runs the layer's forward computation.

The output has shape `(2, 1)` because there are 2 examples and 1 output per example.

`net.weight` and `net.bias` are learnable parameters registered inside the layer.

#### 5. Connection to ML systems

This replaces the manual `linreg(X, w, b)` function and separate parameter tensors.

#### 6. Common confusion points

- A layer initializes parameters before training.
- Registered parameters can be found by `net.parameters()`.
- Output shape `(batch_size, 1)` may differ from label shape `(batch_size,)`.
- Concise does not mean conceptually different.


### 3.5.2 Defining the Loss Function

#### 1. Intuition

A loss module is a framework object that computes loss.

Mean squared error loss measures average squared difference between predictions and labels.

#### 2. Why this exists

Using a built-in loss avoids rewriting common formulas and gives consistent reduction options.


#### 3. Examples

Use PyTorch's mean squared error loss.


In [ ]:
loss_fn = torch.nn.MSELoss()
y_hat = torch.tensor([[1.0], [3.0]])
y = torch.tensor([[2.0], [1.0]])
loss = loss_fn(y_hat, y)
loss


#### 4. Step-by-step breakdown

`torch.nn.MSELoss()` creates a loss object.

Calling `loss_fn(y_hat, y)` compares predictions and labels.

By default, this returns the mean squared error as one scalar.

The prediction and label shapes are both `(2, 1)`, so no accidental broadcasting is needed.

#### 5. Connection to ML systems

This replaces the manual squared-loss function and explicit `.mean()` reduction.

#### 6. Common confusion points

- Built-in losses still need correctly shaped inputs.
- MSE means mean squared error.
- A loss object can hide reduction details, so check defaults.
- The loss is still just a tensor computed from predictions and labels.


### 3.5.3 Defining the Optimization Algorithm

#### 1. Intuition

A PyTorch optimizer stores references to parameters and updates them using gradients.

SGD is stochastic gradient descent, the same update idea used in the from-scratch version.

#### 2. Why this exists

The optimizer object handles repeated parameter updates and gradient-clearing patterns more reliably than handwritten update code.


#### 3. Examples

Create an SGD optimizer for the model parameters.


In [ ]:
net = torch.nn.Linear(2, 1)
trainer = torch.optim.SGD(net.parameters(), lr=0.03)
list(net.parameters())[0].shape


#### 4. Step-by-step breakdown

`net.parameters()` returns the trainable parameters stored in the layer.

`torch.optim.SGD(...)` creates an optimizer that will update those parameters.

`lr=0.03` sets the learning rate.

The optimizer does not update anything until `.step()` is called after gradients exist.

#### 5. Connection to ML systems

This replaces the manual `sgd([w, b], lr, batch_size)` function.

#### 6. Common confusion points

- Creating an optimizer does not train the model.
- The optimizer needs parameters, not predictions.
- `.step()` updates parameters.
- `.zero_grad()` clears old gradients before the next backward pass.


### 3.5.4 Training

#### 1. Intuition

Concise training still follows the same order: batch, prediction, loss, backward, update.

The difference is that PyTorch objects perform some details for us.

#### 2. Why this exists

This version is closer to real PyTorch code while still small enough to trace line by line.


#### 3. Examples

Train a linear layer on synthetic data.


In [ ]:
torch.manual_seed(0)
X = torch.randn(20, 2)
y = (X @ torch.tensor([2.0, -3.4]) + 4.2).reshape(-1, 1)
loader = torch.utils.data.DataLoader(torch.utils.data.TensorDataset(X, y), batch_size=5, shuffle=True)
net = torch.nn.Linear(2, 1)
loss_fn = torch.nn.MSELoss()
trainer = torch.optim.SGD(net.parameters(), lr=0.03)
for Xb, yb in loader:
    trainer.zero_grad(); loss_fn(net(Xb), yb).backward(); trainer.step()
loss_fn(net(X), y)


#### 4. Step-by-step breakdown

The synthetic labels follow a known linear rule.

The DataLoader returns minibatches.

`trainer.zero_grad()` clears gradients from the previous batch.

`loss_fn(net(Xb), yb)` computes the scalar batch loss.

`.backward()` computes gradients.

`trainer.step()` updates the layer parameters.

#### 5. Connection to ML systems

This is the practical PyTorch version of the from-scratch loop. The conceptual order is unchanged.

#### 6. Common confusion points

- Semicolons keep this tiny example compact, but production code should usually use separate lines for readability.
- One pass over the loader is one epoch here.
- Built-in optimizers still require explicit gradient clearing.
- A final loss after one epoch may still be high.


### 3.5.5 Summary

#### 1. Intuition

The concise implementation replaces manual pieces with PyTorch objects.

Model: `torch.nn.Linear`. Loss: `torch.nn.MSELoss`. Optimizer: `torch.optim.SGD`. Data: `DataLoader`.

#### 2. Why this exists

These objects reduce boilerplate once the manual loop is understood.


#### 3. Examples

Map concise objects to from-scratch pieces.


In [ ]:
mapping = {
    "torch.nn.Linear": "linreg plus parameters",
    "torch.nn.MSELoss": "squared_loss plus reduction",
    "torch.optim.SGD": "sgd update function",
    "DataLoader": "data_iter function",
}
mapping


#### 4. Step-by-step breakdown

Each PyTorch object corresponds to a manual concept.

The concise version is shorter because common patterns are packaged.

The package does not remove the underlying sequence of computation.

#### 5. Connection to ML systems

This mapping is the main skill needed to read higher-level training code without losing the execution flow.

#### 6. Common confusion points

- Framework objects are wrappers around concepts you already saw manually.
- Shorter code can be harder to debug if you cannot map it back.
- Always know which object owns parameters.
- Always know when gradients are computed and cleared.


### 3.5.6 Exercises

#### 1. Intuition

These exercises check whether you can modify concise PyTorch training code safely.

#### 2. Why this exists

Real work often means changing model dimensions, learning rates, batch sizes, or loss reductions.


#### 3. Examples

Exercise 1: create a linear model with three input features.


In [ ]:
net3 = torch.nn.Linear(3, 1)
X3 = torch.randn(4, 3)
y3_hat = net3(X3)
y3_hat.shape


Exercise 2: inspect optimizer parameter groups.


In [ ]:
opt = torch.optim.SGD(net3.parameters(), lr=0.01)
len(opt.param_groups), opt.param_groups[0]["lr"]


#### 4. Step-by-step breakdown

Exercise 1 checks input-output dimensions.

Exercise 2 shows that optimizers store parameter groups and settings.

A parameter group is a collection of parameters updated with the same optimizer settings.

#### 5. Connection to ML systems

Parameter groups become useful when different parts of a model need different learning rates.

#### 6. Common confusion points

- The first argument to `Linear` is input feature count.
- The second argument to `Linear` is output count.
- Optimizer settings live in parameter groups.
- Changing a learning rate changes update size, not the model formula.
